# **Atividade Prática**
<font size=3>

- **Tema:** *word-embedding*.
- **Prazo de entrega:** 25 de Maio.

**Envie** o notebook **executado** em formato **ipynb** pelo [formulário](https://docs.google.com/forms/d/e/1FAIpQLSfhkf8HoNNsr9WixEVVlxh8-pFK-rnXsLKN_OLRH_Tg5-5SmA/viewform?usp=sharing&ouid=111377632325147218671).

---

## **Questão 1:**
<font size=3>

Com base no *dataset* de [Transcrição de TED Talks](https://www.kaggle.com/datasets/miguelcorraljr/ted-ultimate-dataset), disponível no diretório $\text{dataset/}\,$, realize os seguintes passos para resolver a **tarefa de regressão**:

### **1º Passo:**
<font size=3>

- Importe o *dataset* e defina o atributo `description` como a lista de dados textuais e `views` como a variável alvo para regressão;
- **Imprima na tela** o tamanho mínimo, o máximo e a média do comprimento das sentenças.


In [ ]:
#from google.colab import drive
#drive.mount('/content/drive/')

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from keras import layers, Model
import tensorflow as tf

In [4]:
print( os.getcwd() )
df = pd.read_csv('./dataset/ted_talks.csv')
df.head()
#df.describe()
#df.info()


/Users/gilcesarf/git/repositories/imd/imd1107-202601/3-unidade


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,native_lang,available_lang,comments,duration,topics,related_talks,url,description,transcript
0,1,Averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,2006-02-25,2006-06-27,TED2006,en,"['ar', 'bg', 'cs', 'de', 'el', 'en', 'es', 'fa...",272.0,977,"['alternative energy', 'cars', 'climate change...","{243: 'New thinking on the climate crisis', 54...",https://www.ted.com/talks/al_gore_averting_the...,With the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre..."
1,92,The best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,2006-02-22,2006-06-27,TED2006,en,"['ar', 'az', 'bg', 'bn', 'bs', 'cs', 'da', 'de...",628.0,1190,"['Africa', 'Asia', 'Google', 'demo', 'economic...","{2056: ""Own your body's data"", 2296: 'A visual...",https://www.ted.com/talks/hans_rosling_the_bes...,You've never seen data presented like this. Wi...,"About 10 years ago, I took on the task to teac..."
2,7,Simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,2006-02-24,2006-06-27,TED2006,en,"['ar', 'bg', 'de', 'el', 'en', 'es', 'fa', 'fr...",124.0,1286,"['computers', 'entertainment', 'interface desi...","{1725: '10 top time-saving tech tips', 2274: '...",https://www.ted.com/talks/david_pogue_simplici...,New York Times columnist David Pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun..."
3,53,Greening the ghetto,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664069,2006-02-26,2006-06-27,TED2006,en,"['ar', 'bg', 'bn', 'ca', 'cs', 'de', 'en', 'es...",219.0,1116,"['MacArthur grant', 'activism', 'business', 'c...",{1041: '3 stories of local eco-entrepreneurshi...,https://www.ted.com/talks/majora_carter_greeni...,"In an emotionally charged talk, MacArthur-winn...",If you're here today — and I'm very happy that...
4,66,Do schools kill creativity?,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65051954,2006-02-25,2006-06-27,TED2006,en,"['af', 'ar', 'az', 'be', 'bg', 'bn', 'ca', 'cs...",4931.0,1164,"['children', 'creativity', 'culture', 'dance',...","{865: 'Bring on the learning revolution!', 173...",https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson makes an entertaining and pro...,Good morning. How are you? (Audience) Good. It...


### **2º Passo:**
<font size=3>

- Realize a normalização da variável alvo (considerando sua adequação à função de atividação da camada de saída);
- Defina o objeto da vetorização dos dados textuais com a função [`TextVectorization`](https://keras.io/api/layers/preprocessing_layers/text/text_vectorization/):
    - A variável `standardize` tem como *default* `"lower_and_strip_punctuation"`, mas esta variável pode receber um pré-processador customizado com *regex*, por exemplo:
    
  <font size=2>
  
  ```python
  
        import tensorflow.strings as tf_strings
        
        def Preprocessor(x):
        
            x = tf.strings.lower(x)
            x = tf.strings.regex_replace(x, r"[^\w\s]", "")
    
            return x
            
    ```
    <font size=3>
  
    - Defina o tamanho do vocabulário (`vocab_size`), o tamanho da entrada/janela do modelo (`max_len`), e a dimensão do vetor *embedding* (`embed_dim`);
    - Iremos realizar a vetorização (inexação) dentro da rede neural, ou seja, o objeto da vetorização irá processar os dados após a camada `layers.Input()`.
    <br>

- Divida os dados em treninamento (90%) e teste (10%). Os dados de validação serão divididos na função `model.fit()`.
  

### **3º Passo:**
<font size=3>

- Desenvolva a primeira versão da **arquitetura neural**:
    - Já sabemos o tamanho da entrada, o tamanho da saída e sua função de ativação:
 
        <font size=2>
        
        ```python
        x_in = layers.Input(shape=(1, ), dtype=tf.string)
            
        ```
        <font size=3>
        
    - A camada [`Embedding`](https://keras.io/api/layers/core_layers/embedding/) irá retornar um *array* de formato (`max_len`, `embed_dim`), assim, precisamos torná-lo unidimensional para poder ser acoplado à próxima camada. Para isso, utilize:
        - Uma camada de [`Flatten`](https://keras.io/api/layers/reshaping_layers/flatten/);
        - Ou uma camada de [`GlobalAveragePooling1D`](https://keras.io/api/layers/pooling_layers/global_average_pooling1d/).
        <br>

- **Escreva** em uma **célula _markdown_** a diferença entre as camadas `Flatten` e `GlobalAveragePooling1D`;

- Compile o modelo com um otimizador ([`SGR`](https://keras.io/api/optimizers/sgd/), [`RMSprop`](https://keras.io/api/optimizers/rmsprop/), [`Adam`](https://keras.io/api/optimizers/adam/));
- Defina a função de perda e métrica de acordo com a natureza do problema.
  

### **4º Passo:**
<font size=3>

- Treine o modelo com o método `.fit()`, definindo as variáveis:
    - `validation_split`: porcentagem de dados para validação;
    - `epochs`: número de épocas para o treinamento;
    - `batch_size`: tamanho do lote da amostragem dos dados de entrada.
      

### **5º Passo:**
<font size=3>

- Quando definido a melhor rede neural, realize o treinamento final.

### **6º Passo:**
<font size=3>

- Realize a avaliação do modelo com o método `.evaluate()`;
- **Imprima na tela** 10 predições com seus respectivos valores **verdadeiros**.

### **7º Passo:**
<font size=3>

- Salve os parâmetros internos do modelo.

### **8º Passo:**
<font size=3>

- Faça o **relatório da modelagem da arquitetura neural** em uma **célula _markdown_**:
    - Descreva quais foram os tipos de camadas neurais que proporcionaram a melhor performance do modelo;
    - Escreva qual otimizador melhor performou o treinamento.
      

---
## **Questão 2:**
<font size=3>

- Com base na arquitetura neural desenvolvida na **Questão 1**, realize o treinamento *final* e avaliação do modelo considerando a **matriz _embedding_** **GloVe** pré-treinada disponível em $\text{dataset/}$).
    > **Importante:** reinicie o *notebook* antes de retreinar um novo modelo.
    
- **Escreva** em uma **célula _markdown_** qual abordagem apresentou melhor performance. **Justifique** sua hipótese.
   